In [ ]:
# === CABECALHO ACADEMICO (gerado por scripts/apply_header.py) ===
# Universidade Presbiteriana Mackenzie
# Disciplina: Inteligência Artificial
# Projeto: Previsão de desempenho de jogadores da NBA (pontos por jogo)
#
# INTEGRANTES DO GRUPO:
#   - Rodrigo Lucas Mascarenhas Leite Oliveira - TIA 10427925
#   - André Ihsan Ward - TIA 10425684
#
# ARQUIVO: notebooks/02_baselines_e_regressao.ipynb
# SINTESE: Baselines e modelos de regressão: validação temporal e teste.
#
# HISTORICO DE ALTERACOES:
#   data       | autor      | descricao
#   -----------|------------|-------------------------------------------
#   2026-11-15 | Rodrigo    | Versão inicial.
# === FIM DO CABECALHO ===


# 02 - Baselines e modelos de regressão

Objetivo: prever `y_pts` (pontos no jogo) usando só informação pré-jogo, e verificar se modelos supervisionados superam as médias recentes que se usam na prática.

**Protocolo** (código em `src/train.py` e `src/evaluate.py`):
1. hiperparâmetros por validação cruzada **temporal** dentro do treino (2021-22 → 2022-23; 2021-23 → 2023-24);
2. escolha do modelo na **validação** (2024-25);
3. **teste** (2025-26) avaliado uma única vez (`python -m src.evaluate --final`).

Este notebook não retreina a grade inteira (isso está em `python -m src.train`): ele reproduz a validação cruzada de um modelo para mostrar o mecanismo e lê os resultados salvos.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 30)
TV = json.loads(Path("results/metrics/treino_validacao.json").read_text(encoding="utf-8"))
TF = json.loads(Path("results/metrics/test_final.json").read_text(encoding="utf-8"))

## 1. Dados e divisão temporal

In [ ]:
from src.modeling import load_dataset, get_split, season_folds, assert_disjoint_temporal, FEATURE_FAMILIES
from src.features import PRE_GAME_FEATURES
df = load_dataset()
assert_disjoint_temporal(df)   # falha se treino/validação/teste se sobrepõem
df.groupby("split").agg(linhas=("y_pts", "size"), inicio=("GAME_DATE", "min"), fim=("GAME_DATE", "max"),
                        temporadas=("SEASON", lambda s: ", ".join(sorted(s.unique())))).loc[["train", "val", "test"]]

In [ ]:
train = get_split(df, "train")
for k, (tr, va) in enumerate(season_folds(train), 1):
    print(f"dobra {k}: treina em {sorted(train.loc[tr, 'SEASON'].unique())} -> valida em {train.loc[va, 'SEASON'].unique()[0]}")
pd.Series({fam: ", ".join(cols) for fam, cols in FEATURE_FAMILIES.items()}, name="features").to_frame()

## 2. Baselines

A baseline prometida no relatório N1 é a média dos 5 jogos anteriores. Ela fica entre as mais fracas das médias recentes (só a de 3 jogos erra mais), então comparamos os modelos também com a média exponencial (fator 0,9) e com uma regressão linear de uma variável sobre ela, que verifica se o ganho vem só de reescalar a média recente. Em MAE, a régua mais exigente é a própria média exponencial e, como diagnóstico, uma regressão **mediana** sobre ela (seção 5).

In [ ]:
from src.modeling import predict_baselines
from src.metrics import regression_metrics
val = get_split(df, "val")
base_val = pd.DataFrame({k: regression_metrics(val["y_pts"], p) for k, p in predict_baselines(val, train).items()}).T
base_val.sort_values("mae")[["mae", "rmse", "r2", "vies"]]

## 3. Validação cruzada temporal (exemplo ao vivo: Ridge)

In [ ]:
from sklearn.model_selection import GridSearchCV
from src.modeling import regression_zoo
pipe, grid = regression_zoo()["ridge"]
busca = GridSearchCV(pipe, grid, cv=season_folds(train), scoring="neg_mean_absolute_error").fit(train[PRE_GAME_FEATURES], train["y_pts"])
print("pipeline:", pipe)
print("melhor alpha:", busca.best_params_, "| MAE médio nas dobras:", round(-busca.best_score_, 3))
print("MAE na validação 2024-25:", round(regression_metrics(val["y_pts"], busca.predict(val[PRE_GAME_FEATURES]))["mae"], 3))

## 4. Todos os modelos: validação cruzada e validação 2024-25 (de `src.train`)

In [ ]:
cv = pd.DataFrame({k: {"MAE CV": -v["score_cv_medio"], "parametros": v["melhores_parametros"], "tempo (s)": TV["regressao"]["tempo_s"][k]}
                   for k, v in TV["regressao"]["cv"].items()}).T
cv["MAE validação"] = [TV["regressao"]["validacao"][k]["mae"] for k in cv.index]
print("modelo final (menor MAE na validação):", TV["regressao"]["modelo_final"])
cv.sort_values("MAE validação")

## 5. Teste 2025-26 (avaliado uma vez)

In [ ]:
t = pd.DataFrame(TF["regressao"]["teste"]).T.sort_values("mae")
t[["tipo", "mae", "rmse", "r2", "vies"]]

In [ ]:
display(Image("results/figures/f_escada_mae.png", width=720))

### Diferenças com intervalo de confiança

Bootstrap pareado **por jogador** (2.000 réplicas): as linhas de um mesmo atleta não são independentes, então o jogador inteiro é a unidade de reamostragem. Delta negativo = o primeiro erra menos.

In [ ]:
fin = TF["regressao"]["modelo_final"]
cmp = {k: v for k, v in TF["regressao"]["comparacoes_bootstrap"].items() if k.startswith(fin + "_vs") or k.startswith("ridge_vs")}
dg = TF["regressao"]["diagnosticos_posteriores"]
cmp[fin + "_vs_mediana_sobre_exponencial (diagnóstico posterior)"] = dg["mediana_sobre_exponencial"]["modelo_final_vs_esta"]
print("MAE da regressão mediana sobre a média exponencial:", round(dg["mediana_sobre_exponencial"]["mae_teste"], 3))
pd.DataFrame(cmp).T[["delta", "ic_inf", "ic_sup", "p_valor"]]

### Resultado esperado (c): a média exponencial supera as médias simples?

In [ ]:
pd.DataFrame(TF["regressao"]["exponencial_vs_simples"]).T[["delta", "ic_inf", "ic_sup", "p_valor"]]

## 6. Erro por faixa de minutos e por temporada

In [ ]:
faixas = {k: pd.DataFrame(v).set_index("grupo")[["n", "mae", "mae_relativo"]] for k, v in TF["regressao"]["por_faixa_de_minutos"].items()}
pd.concat(faixas, axis=1)

In [ ]:
display(Image("results/figures/f_mae_por_faixa_minutos.png", width=640))
display(Image("results/figures/f_mae_por_temporada.png", width=640))

In [ ]:
display(Image("results/figures/f_previsto_vs_observado.png", width=520))

## Leitura

- Todos os modelos superam a baseline da N1 (média de 5 jogos) com folga e intervalo de confiança longe de zero.
- Contra a média exponencial o ganho é pequeno (cerca de 1%), mas estatisticamente consistente. Parte dele vem da perda absoluta: contra uma regressão mediana sobre a média exponencial, o ganho cai para cerca de 0,04 ponto, que é a informação que os modelos acrescentam além da média recente.
- O gradient boosting com perda absoluta vence porque otimiza diretamente o MAE: com pontos assimétricos à direita, a mediana erra menos em valor absoluto que a média.
- O erro absoluto cresce com os minutos (quem joga mais pontua mais e oscila mais), mas o erro **relativo** cai.
- O desempenho é estável entre temporadas: não há sinal de deriva forte (os valores de 2022-23 e 2023-24 vêm das dobras usadas para escolher hiperparâmetros e são levemente otimistas).